# MACURA races a drone
A drone races laps around a fixed 3-D course: four gates at 1-4 m, a near-vertical 3 m **chute**, two pillars,
a steady wind (different every flight, the drone knows it) and a package of random weight. Every metre of
course flown pays, so **flying faster always pays more** - but dropping the chute too fast **while the rotors
push** makes the drone lose thrust (the vortex ring state) and crash. We compare four reinforcement learning
algorithms on the exact same task, changing only one thing between them: how each uses its learned world model
to imagine extra practice data. MACURA, then MBPO, M2AC and plain SAC, one at a time, then side by side. Runs on
a Kaggle GPU kernel, top to bottom.

## **Get the code**
Clone the project code from GitHub into the kernel. Turn on Internet in the kernel settings and
add a Secret named GITHUB_TOKEN (Add-ons, Secrets) for the private repo.

In [ ]:
import os, sys, subprocess

REPO   = "silvergjeka22/macura-drone"
BRANCH = "cage-env"       # the RACE experiment lives on this branch (main = ring study)
ROOT   = "/kaggle/working/macura-drone"

# The task is chosen BEFORE the config is imported (setup() imports it): the RACE, 1 seed (0), 50k steps.
os.environ["MACURA_TASK"] = "race"
os.environ["MACURA_STEPS"] = "10000"     # TEST RUN: 10k steps. Delete this line for the full 50k run.
# Optional overrides: os.environ["MACURA_SEEDS"] = "1 2"; os.environ["MACURA_ALGOS"] = "macura mbpo";
#   os.environ["MACURA_STEPS"] = "2000" (quick test). Tuning (paper protocol): os.environ["MACURA_XI"] = "0.5" | "1" | "2";
#   os.environ["MBPO_HORIZON"] = "5"; os.environ["MACURA_EXPLORATION"] = "equal"  (same pink noise for all)

# GitHub token from a Kaggle Secret (or an env var if you set one)
token = os.environ.get("GITHUB_TOKEN")
if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://{token}@github.com/{REPO}.git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

## **Setup**
Install the packages the kernel needs, pick a safe graphics backend, and make the output folders
under /kaggle/working/runs.

In [ ]:
from src.bootstrap import setup
setup(root=ROOT)

In [ ]:
# load every project function into the notebook
%run $ROOT/src/imports.py
set_seed(cfg.SEED)

## **Fresh start**
Wipe old checkpoints, logs, plots and videos so nothing stale is reused.

In [ ]:
import shutil
for d in ("checkpoints", "videos", "plots", "logs"):
    shutil.rmtree(f"{cfg.OUTPUT_ROOT}/{d}", ignore_errors=True)
    os.makedirs(f"{cfg.OUTPUT_ROOT}/{d}", exist_ok=True)
print("cleared old runs:", cfg.OUTPUT_ROOT)

## **Meet the course**
The race flown by a small **hand-written autopilot** (NOT one of the learning algorithms), three ways, on the
same three flights: **careful** (1.5 m/s), **fast** (3 m/s, drops the chute at a controlled 1.1 m/s) and
**chute dive** (3 m/s, drops the chute at 3 m/s). The fast one is the best way to fly it; the dive loses thrust
in the chute and crashes. Orange rings = gates, red column = the chute, the windsock shows the wind, the trail
turns red while the drone loses thrust (the drone is drawn 2x larger so it stays visible). The four algorithms
have to learn all of this from scratch.

In [ ]:
path = record_race_video(cfg.CFG, {"Careful 1.5 m/s": "autopilot:0.8:1.5", "Fast 3 m/s": "autopilot:1.1:3.0",
                                   "Chute dive": "autopilot:3.0:3.0"},       # hand-written demo controller, NOT learned
                         f"{cfg.OUTPUT_ROOT}/videos/course_demo.mp4", seeds=(6027, 6041, 6015))
if path:
    display(Video(path, embed=True, width=900))

## **How the environment works**
The same task for all four algorithms, built from a few pieces (figures below):

- **Course** — a 6 x 6 m rounded square flown clockwise through four gates: **P1 2 m → P2 4 m → the chute (a near-vertical 3 m drop) → P3 1 m → P4 1.5 m → P1** (24.3 m per lap). The drone starts at a random point of the course; a flight lasts 10 s.
- **Reward** — every step pays the drone's **speed along the course** (only while it stays near it), minus the distance outside a 0.35 m tube. There is **no finish line**: faster = more laps = more reward, like "run as fast as you can" in the MACURA paper's tasks. Staying upright pays only 0.1 per step, so hovering is worth almost nothing.
- **Crash** — flipping, touching the ground, hitting a pillar or flying more than 1.5 m off the course ends the flight and **costs 500** (about 10 m of course).
- **Lift loss (powered vortex ring state)** — sinking **straight down along the rotor axis faster than 1.2 m/s while the rotors push** costs thrust, up to **35% at 2.2 m/s**. That is exactly what happens if the chute is dropped too fast.
- **Wind** — a **steady wind** of random direction and strength (up to 0.5 N) every flight, **visible** to the drone, plus small hidden gusts and actuator noise.
- **Package** — **0 to 0.2 kg**, different every flight, **visible** to the drone.
- **Pillars** — two, inside the corners after the chute and before P1: cutting those corners crashes.

**Checked before any training** (hand-written autopilot, 30 flights each): 3 m/s with a controlled chute flies **0.93 laps per flight with no crash**; the same speed dropping the chute at 3 m/s crashes **83%**; 1.5 m/s flies 0.52 laps. With the agents' discount (gamma 0.99): fast & safe 209, dive 144, slow 114, hovering 10 - racing pays, diving does not.

*Why this task:* the danger only appears when a policy starts to go fast, so it is **new** to the learned model at that point, and it keeps moving as the policy improves. Checked offline: a model trained on flights up to 2.2 m/s disagrees with itself **16-20x** more and is **2.5x** more wrong when racing faster, and **150-190x / 3.3-3.8x** when dropping the chute fast. A fixed-horizon method (MBPO) keeps imagining there; MACURA stops where its models disagree. Whether that makes MACURA faster and safer is exactly what this run measures.

In [ ]:
# how the task is built (the SAME env every algorithm learns on)
plot_race_course(cfg.CFG, save_path=f"{cfg.OUTPUT_ROOT}/plots/env_course.png")        # the course + its heights
plot_lift_loss(cfg.CFG, save_path=f"{cfg.OUTPUT_ROOT}/plots/env_lift_loss.png")        # the danger in the chute
plot_wind_process(cfg.CFG, save_path=f"{cfg.OUTPUT_ROOT}/plots/env_wind.png")          # hidden gusts + actuator noise
plot_env_difficulty(cfg.CFG, save_path=f"{cfg.OUTPUT_ROOT}/plots/env_difficulty.png"); # how hard the raw env is

## **How the comparison works**
All four algorithms share the same policy learner (SAC), the same kind of learned world model (7 networks,
the 5 best used) and the same evaluation scenarios. This run follows the **MACURA paper's protocol**:
- SAC uses no model (it learns only from real flights).
- MBPO imagines fixed-length rollouts.
- M2AC imagines fixed-length rollouts and, at every step, keeps only the least uncertain part (exactly as in the M2AC paper).
- MACURA imagines rollouts but stops each one as soon as its models disagree.

Every model-based method gets 100 imagined rollouts per real step and trains on 95% imagined / 5% real data.

**Exploration, reported openly:** as in the MACURA paper, MACURA explores with pink noise while MBPO and
M2AC act deterministically (SAC samples from its own policy). The paper itself shows exploration changes
results, so this is a known confound. Set `MACURA_EXPLORATION=equal` to give all four the same pink noise.

## **MACURA**
MACURA trusts the model only where the model trusts itself. Step by step:
1. Imagine short rollouts from real states, using the learned world model.
2. At each step, measure how much the ensemble of models disagree (its uncertainty).
3. Keep imagining while the uncertainty stays low, and stop that rollout the moment it gets
   high. Safe regions get long rollouts, risky regions get short ones.
4. Train the SAC policy on this clean imagined data plus a little real data.

This is the method we want to test.

In [ ]:
macura_runs = train_algo("macura", cfg.CFG, cfg.OUTPUT_ROOT)

In [ ]:
plot_sample_efficiency(macura_runs, f"{cfg.OUTPUT_ROOT}/plots/macura_return.png")
plot_failure_rate(macura_runs, f"{cfg.OUTPUT_ROOT}/plots/macura_crash.png");
plot_rollout_depth(best_run(macura_runs), f"{cfg.OUTPUT_ROOT}/plots/macura_rollout_depth.png");
# where MACURA stops trusting its model: fast descents (the chute) vs the rest
plot_trust_by_zone(macura_runs, f"{cfg.OUTPUT_ROOT}/plots/macura_trust_by_descent.png", split="descent");

Test: fly the best MACURA policy and look at its numbers and a short clip.

In [ ]:
b = best_run(macura_runs)
print("MACURA best seed", b["seed"],
      "-> return", round(b["final_eval"]["eval_return"], 1),
      "| crash", round(b["final_eval"]["eval_failure_rate"], 2),
      "| laps per flight", round(b["final_eval"].get("eval_laps", float("nan")), 2),
      "| full lap", round(b["final_eval"]["eval_success_rate"], 2))
# ~1-min highlight reel: many flights, the best ones stitched with per-clip captions
macura_video = record_policy_video(b["checkpoint"], cfg.CFG, f"{cfg.OUTPUT_ROOT}/videos/macura.mp4",
                                   seconds=60, label="MACURA")
if macura_video:
    display(Video(macura_video, embed=True, width=440))

## **MBPO**
MBPO is the baseline MACURA builds on. Step by step:
1. Imagine rollouts from real states, using the world model.
2. Use a fixed rollout length from a schedule (it does not look at uncertainty).
3. Train SAC on all of that imagined data plus a little real data.

Because the length is fixed, in risky regions MBPO keeps imagining past the point where the
model is reliable, which can feed the policy misleading data.

In [ ]:
mbpo_runs = train_algo("mbpo", cfg.CFG, cfg.OUTPUT_ROOT)

In [ ]:
plot_sample_efficiency(mbpo_runs, f"{cfg.OUTPUT_ROOT}/plots/mbpo_return.png")
plot_failure_rate(mbpo_runs, f"{cfg.OUTPUT_ROOT}/plots/mbpo_crash.png");

Test: fly the best MBPO policy and look at its numbers and a short clip.

In [ ]:
b = best_run(mbpo_runs)
print("MBPO best seed", b["seed"],
      "-> return", round(b["final_eval"]["eval_return"], 1),
      "| crash", round(b["final_eval"]["eval_failure_rate"], 2),
      "| laps per flight", round(b["final_eval"].get("eval_laps", float("nan")), 2),
      "| full lap", round(b["final_eval"]["eval_success_rate"], 2))
mbpo_video = record_policy_video(b["checkpoint"], cfg.CFG, f"{cfg.OUTPUT_ROOT}/videos/mbpo.mp4",
                                 seconds=60, label="MBPO")
if mbpo_video:
    display(Video(mbpo_video, embed=True, width=440))

## **M2AC**
M2AC also imagines fixed length rollouts, then filters them. Step by step:
1. Imagine fixed length rollouts.
2. Score each imagined step by how much the models disagree.
3. Keep only the most trustworthy fraction and drop the rest.
4. Train SAC on what is left plus a little real data.

It reacts to uncertainty by filtering after the fact, instead of stopping early.

In [ ]:
m2ac_runs = train_algo("m2ac", cfg.CFG, cfg.OUTPUT_ROOT)

In [ ]:
plot_sample_efficiency(m2ac_runs, f"{cfg.OUTPUT_ROOT}/plots/m2ac_return.png")
plot_failure_rate(m2ac_runs, f"{cfg.OUTPUT_ROOT}/plots/m2ac_crash.png");

Test: fly the best M2AC policy and look at its numbers and a short clip.

In [ ]:
b = best_run(m2ac_runs)
print("M2AC best seed", b["seed"],
      "-> return", round(b["final_eval"]["eval_return"], 1),
      "| crash", round(b["final_eval"]["eval_failure_rate"], 2),
      "| laps per flight", round(b["final_eval"].get("eval_laps", float("nan")), 2),
      "| full lap", round(b["final_eval"]["eval_success_rate"], 2))
m2ac_video = record_policy_video(b["checkpoint"], cfg.CFG, f"{cfg.OUTPUT_ROOT}/videos/m2ac.mp4",
                                 seconds=60, label="M2AC")
if m2ac_video:
    display(Video(m2ac_video, embed=True, width=440))

## **SAC**
SAC is the model free baseline. It has no world model and never imagines anything: it learns
only from the real flights it collects. It is the reference for how much the world model helps
the other three.

In [ ]:
sac_runs = train_algo("sac", cfg.CFG, cfg.OUTPUT_ROOT)

In [ ]:
plot_sample_efficiency(sac_runs, f"{cfg.OUTPUT_ROOT}/plots/sac_return.png")
plot_failure_rate(sac_runs, f"{cfg.OUTPUT_ROOT}/plots/sac_crash.png");

Test: fly the best SAC policy and look at its numbers and a short clip.

In [ ]:
b = best_run(sac_runs)
print("SAC best seed", b["seed"],
      "-> return", round(b["final_eval"]["eval_return"], 1),
      "| crash", round(b["final_eval"]["eval_failure_rate"], 2),
      "| laps per flight", round(b["final_eval"].get("eval_laps", float("nan")), 2),
      "| full lap", round(b["final_eval"]["eval_success_rate"], 2))
sac_video = record_policy_video(b["checkpoint"], cfg.CFG, f"{cfg.OUTPUT_ROOT}/videos/sac.mp4",
                                seconds=60, label="SAC")
if sac_video:
    display(Video(sac_video, embed=True, width=440))

## **All algorithms compared**
Now all four on the same axes. **Main scores, fixed BEFORE the run:** the **average return over training** (how
fast each method learns to race) and the **real crashes while learning** (drones broken during training, not in
the tests). Second: laps per flight, final return and final crash rate, then stability. The final return
re-tests each method's single best checkpoint on **fresh scenarios** (not the ones it was picked on).

Then the **why**: how much of the imagined data each method trained on was data its own models **disagreed
about** (above MACURA's trust threshold), how much of it was **fast descents** (the chute, where the lift loss
makes the model wrong), and whether the crashes happen with **heavy packages**.

In [ ]:
runs = macura_runs + mbpo_runs + m2ac_runs + sac_runs
plot_overview(runs, f"{cfg.OUTPUT_ROOT}/plots/overview.png")
plot_iqm_efficiency(runs, f"{cfg.OUTPUT_ROOT}/plots/iqm.png")
plot_crash_rate_ci(runs, f"{cfg.OUTPUT_ROOT}/plots/crash_ci.png")
plot_training_crashes(runs, f"{cfg.OUTPUT_ROOT}/plots/training_crashes.png")   # drones broken while learning
plot_race_laps(runs, f"{cfg.OUTPUT_ROOT}/plots/laps.png")                           # laps per 10 s flight
plot_steps_to_threshold(runs, save_path=f"{cfg.OUTPUT_ROOT}/plots/steps_to_threshold.png")
plot_summary_table(runs, f"{cfg.OUTPUT_ROOT}/plots/summary.png")
# WHY: how much untrustworthy imagined data each method trains on (MACURA filters it out)
plot_untrusted_data(runs, f"{cfg.OUTPUT_ROOT}/plots/untrusted_data.png")
# WHY: imagined FAST descents (the chute) each method trains on, and crashes by package weight
plot_fast_data(runs, f"{cfg.OUTPUT_ROOT}/plots/fast_descent_data.png")
plot_crash_by_payload(runs, f"{cfg.OUTPUT_ROOT}/plots/crash_by_payload.png");

## **All videos side by side**
The four best policies in one clip, each labelled, so you can watch the difference directly.

In [ ]:
clips = {"MACURA": macura_video, "MBPO": mbpo_video, "M2AC": m2ac_video, "SAC": sac_video}
clips = {k: v for k, v in clips.items() if v}
if clips:
    reel = stitch_videos_grid(clips, f"{cfg.OUTPUT_ROOT}/videos/reel.mp4", cols=2)
    display(Video(reel, embed=True, width=560))

## **The race: MACURA vs MBPO on the same flights**
Both best policies fly the **same four fresh scenarios** (same start point, package, wind and gusts), side by
side, with a running scoreboard of laps and crashes. Trail dots are green while the drone has full thrust and
red while it is losing it (the drone is drawn 2x larger so it stays visible).

In [ ]:
race = record_race_video(cfg.CFG, {"MACURA": best_run(macura_runs)["checkpoint"],
                                   "MBPO": best_run(mbpo_runs)["checkpoint"]},
                          f"{cfg.OUTPUT_ROOT}/videos/race_macura_mbpo.mp4", seeds=(1000, 1001, 1002, 1003))
if race:
    display(Video(race, embed=True, width=720))

## **Save**
Everything is under /kaggle/working/runs (checkpoints, logs, plots, videos) and is saved as the
kernel output. Download it, then watch a policy live on your Mac with:

`mjpython run_live_mac.py --compare out/runs/checkpoints`